In [ ]:
"""LAB 3 - Prompt caching.
 
Goal: prove a cache write, a cache hit, and a cache MISS (prefix changed) using the
usage fields, and calculate the savings.
 
WHEN TO CACHE: the SAME large prefix (system prompt, tool defs, documents, long history)
is reused across many requests within the cache lifetime (5 min default, 1 h optional).
It cuts cost and latency. It does NOT change the answer.
WHEN NOT TO: prefix is small (below the model minimum it silently isn't cached),
or each request has a different prefix, or requests are far apart in time.
"""


In [43]:
from common import MODEL, banner, get_client
client = get_client()

WRITE_5M = 1.25 # 5-minute cache write
READ = 0.10 # cache read (some newer models are lower, eg. 0.05; check the pricing table)

## build a big, STATIC prefix (~5 - 6k tokens, sbove every model minium cacheble length)

POLICY = "\n".join(
        f"Policy {i}: Section {i} of the company delivery handbook states that every client "
        f"engagement must document risk item {i}, assign an owner, and reveiw it weekly."
        for i in range(1,220))


##Shrink Policy

POLICY_SHRINK = "\n".join(
        f"Policy {i}: Section {i} of the company delivery handbook states that every client "
        f"engagement must document risk item {i}, assign an owner, and reveiw it weekly."
        for i in range(1,5))

In [44]:
def ask(question:str, header:str = "You are a delivery-policy assisant.") -> dict:
    resp = client.messages.create(
        model=MODEL,
        max_tokens = 80,
        system = [{"type":"text","text":header},
        ##Breakpoint : Everything until this point and inclusing this block becomes the cached prefix
        {"type":"text","text":POLICY_SHRINK, "cache_control":{"type":"ephemeral"}}],
        messages = [{"role":"user","content":question}],
    )
    u = resp.usage
    return {"uncached_in":u.input_tokens,
    "cache_write":u.cache_creation_input_tokens or 0,
    "cache_read":u.cache_read_input_tokens or 0,
    "out":u.output_tokens}
    # "response": resp.content[0].text}

def cost_units(u:dict) -> float:
    """input cost in 'normal input token' units."""
    return u["uncached_in"] + WRITE_5M * u["cache_write"] + READ * u["cache_read"]


In [45]:
banner("Step1 : First call --> Expect cache a write")
a = ask("what must engagement document")
print(a, "->cost units:",round(cost_units(a)))


======== Step1 : First call --> Expect cache a write ========
{'uncached_in': 235, 'cache_write': 0, 'cache_read': 0, 'out': 80} ->cost units: 235


In [46]:
banner("Step2: Same prefix, new question -> expect a cache read")
b = ask("who is assigned to risk item 7")
print(b,"->cost units:",round(cost_units(b)))


======== Step2: Same prefix, new question -> expect a cache read ========
{'uncached_in': 240, 'cache_write': 0, 'cache_read': 0, 'out': 80} ->cost units: 240


In [17]:
banner("Step3: Chnage the FIRST block -> prefix differs -> expect a MISS (new write)")
c = ask("what must every engagement document?", header="You are a DIFFERENT assistant.")
print(c,"->cost units:",round(cost_units(c)))


======== Step3: Chnage the FIRST block -> prefix differs -> expect a MISS (new write) ========
{'uncached_in': 14, 'cache_write': 11404, 'cache_read': 0, 'out': 80} ->cost units: 14269


In [41]:
d = ask("i")
d

{'uncached_in': 23, 'cache_write': 0, 'cache_read': 0, 'out': 80}

In [30]:
banner ("Final Step: Calculation of Cache and withour cache difference")
no_cache = a['cache_write'] + a["uncached_in"]

print(f"\nWithout caching call B would cost ~{no_cache} units: with caching {round(cost_units(b))}.")
print("Savings on call B: ~%.0f%%" % (100 * (1 - cost_units(b) / no_cache)))


======== Final Step: Calculation of Cache and withour cache difference ========

Without caching call B would cost ~11415 units: with caching 1157.
Savings on call B: ~90%


In [ ]:
""" Check Yourself 

1. Total Input = uncached_in + cache_write + cache_read. does it match across Step# 1/2/3?
    Answer: Yes it is match across all cases
2. Why did Step3 miss even though the big POLICY block was identical? (Prefix order Matters)?
    Answer: Because the system prompt is different from earlier. This will trigger agent to search again. 
3. if both cache_write and cache_read are 0, what is the most likely reason?
    Answer: Cache control was not enabled or the content is below the minimum token threshold. If enabled both should not be zero always. Either should have values. 
        eg: Comment the cache code in ask function and run ask func with some question
4. Experiment: shrink Policy to range (1,5) what happen? Do you get an error?
    Answer: No cache captured. Both Write and Read is Zero because below the minimum token threshold. 

"""



11415